# Spatial Join


In the previous section, we looked at how to determine spatial relationships between features using spatial predicates. Now let's turn to an operation that puts those relationships to work — **combining attribute data from two layers based on their spatial positions relative to one another**.

A **spatial join** links the attributes of two spatial datasets according to how their geometries relate.

With a spatial join, you can, for example:

- determine which district each point belongs to;
- aggregate data by spatial unit.

In GeoPandas, this is done using the `sjoin` function, which lets you specify both the spatial predicate and the type of join.

In this section, we will walk through how to perform spatial joins and use them to analyse geodata.


## 0. Importing Libraries and Preparing the Data


### 0.1. Importing Libraries


In [ ]:
import pandas as pd
import geopandas as gpd

### 0.2. Preparing the Data


This section uses two files from `data/vienna/`:

- **vienna_admin.gpkg** — boundaries of the city's districts and census districts;
- **vienna_top_locations.csv** — well-known places to visit in Vienna.

_Every dataset and its source is listed on the [Course Modules](../module_0/syllabus.md) page._


We read the district boundaries of Vienna from the GeoPackage file. The city is divided into 23 numbered districts, each with a name of its own.


In [ ]:
districts = gpd.read_file("../../data/vienna/vienna_admin.gpkg", layer="district")

districts.explore(tiles="cartodbpositron")

Then the top locations from the CSV file, turned into a `GeoDataFrame`.


In [ ]:
locations_csv = pd.read_csv("../../data/vienna/vienna_top_locations.csv", sep=";", decimal=",")
locations_csv = locations_csv.dropna(subset=["geo_longitude", "geo_latitude"])

locations_gdf = gpd.GeoDataFrame(
    locations_csv,
    geometry=gpd.points_from_xy(locations_csv["geo_longitude"], locations_csv["geo_latitude"]),
    crs="EPSG:4326"
)

locations_gdf.explore(tiles="cartodbpositron")

## 1. Joining the Data


Let's determine which district each location is in. To do this, we will perform a spatial join between the locations layer and the districts layer.

Recall that a spatial join relies on a spatial predicate. Here, we need to find **which district each location falls inside**, so we will use the `within` predicate.

Before running the join, make sure both layers share a CRS. If they differ, reproject one of them first.


### 1.1. Checking the CRS


Do the two layers share a CRS?


In [ ]:
districts.crs == locations_gdf.crs

They match, so we can proceed without reprojecting.


### 1.2. Performing the Spatial Join


For each location, we want to identify the district it falls within. We use `sjoin` with the `within` predicate and a left join (`how="left"`) to retain all locations, including any that do not fall within a district boundary.


In [ ]:
locations_in_district = gpd.sjoin(
    locations_gdf,
    districts,
    how="left",
    predicate="within"
)

The spatial join appends the attributes of the district to each location feature, based on which district it falls within. Locations that fall outside every district keep their own attributes and get `NaN` in the joined columns — let's check whether there are any:

In [ ]:
locations_in_district["NAME"].isna().sum()

Every location fell inside a district, so nothing was lost.

Now the result. We'll display location names alongside their district names, using the field names as they appear in the source data: `title` for locations and `NAME` for districts.

In [ ]:
locations_in_district[["title", "NAME"]].head()

We now have information on which district each location is in.


## 2. Aggregating the Results


Now that the spatial join has assigned a district to each location, we can aggregate the results — for example, to count how many locations fall in each district.

Let's group the data by district name (`NAME`) and count the number of locations in each. The result is a table showing the count per district, sorted in descending order.


In [ ]:
location_counts = (
    locations_in_district
    .groupby("NAME")
    .size()
    .reset_index(name="location_count")
    .sort_values("location_count", ascending=False)
)

location_counts.head()

In this way, a spatial join allows us to move from analysing individual features to analysing spatial units.


## Summary


In this section, we covered the spatial join — a method for linking data from two layers based on their spatial relationship.

We saw how the `sjoin` function can be used to match features across layers using a spatial predicate.

We also demonstrated how the result of a spatial join can feed directly into further analysis — such as aggregating data and counting features within defined spatial units.
